# Bronze ingestion test

Ingests `data/B001/customer_B001.csv` into a bronze table using the `pipeline` functions directly. The cells show:
- a first load (all rows inserted)
- a rerun of the same file (idempotency: every row is a replay, nothing inserted)
- a failed run, with the error naming the step

In [1]:
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

# Make src/ importable when the notebook runs from anywhere inside the project
root = Path.cwd()
while not (root / 'src').exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / 'src'))

from pyspark.sql.types import StructType, StructField, StringType
from pipeline.session import get_spark_session
from pipeline.catalog import ensure_schema_exists, get_schema_fqn, get_table_fqn
from pipeline.audit import AuditRun
from pipeline.errors import step
from pipeline.metadata import add_metadata
from pipeline.reader import read_source_csv
from pipeline.validation import assert_columns_exist, count_duplicate_keys, count_null_business_keys
from pipeline.writer import insert_new_rows
from bronze.entity import EntityConfig

spark = get_spark_session('bronze_ingestion_test')

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/home/charlescarpenter/.local/share/Project_Mercury/cacerts -Djavax.net.ssl.trustStorePassword=changeit
Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/home/charlescarpenter/.local/share/Project_Mercury/cacerts -Djavax.net.ssl.trustStorePassword=changeit
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/11 12:45:55 WARN Utils: Your hostname, DP41ZL4, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/11 12:45:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/charlescarpenter/venvs/mercury-delta/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/charlescarpenter/.ivy2.5.2/cache
The jars for the packages stored in: /home/charlescarpenter/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dep

In [2]:
import logging
from pipeline.events import EventBus
from pipeline.log import make_log_subscriber

logging.basicConfig(format='%(asctime)s %(levelname)s %(message)s', level=logging.INFO)

bus = EventBus()
bus.subscribe(make_log_subscriber(logging.getLogger('pipeline')))

1

In [3]:
COLUMNS = [
    'source_record_id', 'customer_id', 'first_name', 'last_name', 'email', 'phone',
    'date_of_birth', 'loyalty_status', 'marketing_opt_in', 'created_at', 'updated_at',
]
# Bronze keeps every source column as a string
SCHEMA = StructType([StructField(c, StringType(), True) for c in COLUMNS])

PROJECT_DATA = root / 'data' / 'B001'
CUSTOMER_FILE = PROJECT_DATA / 'customer_B001.csv'

CATALOG = 'forge'
TEST_SCHEMA = 'bronze_ingestion_test'
ensure_schema_exists(spark, CATALOG, TEST_SCHEMA)

entity = EntityConfig(
    name='customer',
    source_path=str(CUSTOMER_FILE),
    schema=SCHEMA,
    target_table=get_table_fqn(CATALOG, TEST_SCHEMA, 'customer'),
    merge_keys=('customer_id',),
)

print('source:', entity.source_path)
print('target:', entity.target_table)

source: /home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/customer_B001.csv
target: bronze_ingestion_test.customer


In [4]:
records = []  # the audit sink: AuditRun appends each finished AuditRecord here

def ingest(path: str):
    with AuditRun('bronze', entity.name, entity.target_table, sink=records.append, bus=bus) as run:
        with step('read', bus=bus, path=path):
            df = read_source_csv(spark, entity.schema, path, entity.read_options, file_path_column='_source_file')

        with step('add_metadata', bus=bus):
            df = add_metadata(
                df,
                run_id=run.record.run_id,
                ingested_at=datetime.now(timezone.utc),
                source_columns=[f.name for f in entity.schema.fields],
            )

        with step('validate', bus=bus):
            assert_columns_exist(df, entity.merge_keys)
            run.add('source_row_count', df.count())
            run.add('null_business_key_count', count_null_business_keys(df, entity.merge_keys))
            run.add('duplicate_business_key_count', count_duplicate_keys(df, entity.merge_keys))

        with step('write', table=entity.target_table, bus=bus):
            result = insert_new_rows(spark, df, entity.target_table, entity.merge_keys, hash_column='_record_hash')
            for name, value in asdict(result).items():
                run.add(name, value)

    return records[-1]

def show(record):
    print(record.status, record.metrics, record.error_message or '')

In [5]:
# First load: every row should be inserted
show(ingest(entity.source_path))

2026-10-11 12:46:05,722 INFO bronze customer run 62ee11f6-0ffe-45e7-8762-0a7466f86644 started


2026-10-11 12:46:06,138 INFO step 'read' (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/customer_B001.csv) completed in 0.41s
2026-10-11 12:46:06,396 INFO step 'add_metadata' completed in 0.26s
2026-10-11 12:46:09,635 INFO step 'validate' completed in 3.24s
2026-10-11 12:46:19,001 INFO step 'write' (table=bronze_ingestion_test.customer) completed in 9.36s
2026-10-11 12:46:19,004 INFO bronze customer run 62ee11f6-0ffe-45e7-8762-0a7466f86644 succeeded: {'source_row_count': 50, 'null_business_key_count': 0, 'duplicate_business_key_count': 0, 'replay_existing_row_count': 0, 'inserted_row_count': 50, 'hash_conflict_count': 0, 'target_total_row_count': 50}


success {'source_row_count': 50, 'null_business_key_count': 0, 'duplicate_business_key_count': 0, 'replay_existing_row_count': 0, 'inserted_row_count': 50, 'hash_conflict_count': 0, 'target_total_row_count': 50} 


In [6]:
# Rerun of the same file: every row is a replay, so nothing is inserted
show(ingest(entity.source_path))

2026-10-11 12:46:19,023 INFO bronze customer run 24d0daff-a7de-42c4-aa08-c9a95d2aa64a started
2026-10-11 12:46:19,093 INFO step 'read' (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/customer_B001.csv) completed in 0.07s
2026-10-11 12:46:19,204 INFO step 'add_metadata' completed in 0.11s
2026-10-11 12:46:19,922 INFO step 'validate' completed in 0.72s
2026-10-11 12:46:25,284 INFO step 'write' (table=bronze_ingestion_test.customer) completed in 5.36s
2026-10-11 12:46:25,285 INFO bronze customer run 24d0daff-a7de-42c4-aa08-c9a95d2aa64a succeeded: {'source_row_count': 50, 'null_business_key_count': 0, 'duplicate_business_key_count': 0, 'replay_existing_row_count': 50, 'inserted_row_count': 0, 'hash_conflict_count': 0, 'target_total_row_count': 50}


success {'source_row_count': 50, 'null_business_key_count': 0, 'duplicate_business_key_count': 0, 'replay_existing_row_count': 50, 'inserted_row_count': 0, 'hash_conflict_count': 0, 'target_total_row_count': 50} 


In [7]:
# Table contents: one row per customer_id
spark.read.table(entity.target_table).orderBy('customer_id').show(truncate=False)

+---------------------------+-----------+----------+----------+----------------------------------------+--------------+-------------+--------------+----------------+-------------------+-------------------+------------------------------------------------------------------------------------------+----------------------------------------------------------------+------------------------------------+-------------------------+
|source_record_id           |customer_id|first_name|last_name |email                                   |phone         |date_of_birth|loyalty_status|marketing_opt_in|created_at         |updated_at         |_source_file                                                                              |_record_hash                                                    |_run_id                             |_ingested_at             |
+---------------------------+-----------+----------+----------+----------------------------------------+--------------+-------------+--------------+--

In [8]:
# A missing source file: expect status 'failed' and an error naming the 'read' step
try:
    ingest(str(PROJECT_DATA / 'missing.csv'))
except Exception as exc:
    print('raised:', exc)

show(records[-1])

2026-10-11 12:46:25,917 INFO bronze customer run d274db04-8bda-409b-a62e-7f854e7b90c2 started
2026-10-11 12:46:26,024 ERROR step 'read' (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv) failed: AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv. SQLSTATE: 42K03
2026-10-11 12:46:26,025 ERROR bronze customer run d274db04-8bda-409b-a62e-7f854e7b90c2 failed: Step 'read' failed (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv): AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv. SQLSTATE: 42K03


raised: Step 'read' failed (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv): AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv. SQLSTATE: 42K03
failed {} Step 'read' failed (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv): AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv. SQLSTATE: 42K03


In [9]:
spark.sql(f'DROP SCHEMA IF EXISTS {get_schema_fqn(CATALOG, TEST_SCHEMA)} CASCADE')

DataFrame[]